# Wilson-Cowan Second-order Diffusive (PD) Model - Line Plots

**Model:** Line plots of metrics vs coupling strength K for PD model

## Overview

This notebook generates line plots of key metrics (oscillation presence, transfer entropy, directed transfer entropy, higher-order statistical measures) as functions of coupling strength K for the Second-order Diffusive model. These visualizations help identify relationships between coupling parameters and the emergence of higher-order statistical structure.

## Key Sections

- **Load parameter sweep results:** Read computed metrics from saved arrays
- **Plot metrics vs K:** Visualize relationships between coupling and statistical structure

---

In [4]:
import sys
from tqdm import tqdm
sys.path.append('../')  # Ensure Python can find the 'Scripts' folder

#from Scripts.parameters import *
from Scripts.parameters_random import *
from Scripts.dynamics import *
from Scripts.simulation import *
from Scripts.oscillation_detection import *
from Scripts.metrics import *
from Scripts.visualization import *

# Parameter ranges for oscillation detection
P_values = np.linspace(1.0, 10.0, 20)
K_values = np.linspace(0.0, 1.0, 20)
K3_values = np.linspace(0.0, 1.0, 20)

print("Functions loaded successfully.")

Functions loaded successfully.


---
## **4. Oscillation Detection in $(P \times K$) Space**
We perform a parameter sweep to detect oscillations in the $(P \times K$) space.


In [6]:
P_values = [3,4,5,6]
K3_values = np.linspace(0.0, 1.0, 20)

# Matrix to store oscillation detection
oscillation_map = np.zeros((len(P_values), len(K_values)), dtype=int)

# Matrices to store metrics with noise
TC_map = np.zeros((len(P_values), len(K_values)))
DTC_map = np.zeros((len(P_values), len(K_values)))
Cumulant_map = np.zeros((len(P_values), len(K_values)))
PowerCorr_map = np.zeros((len(P_values), len(K_values)))
Entropy_map = np.zeros((len(P_values), len(K_values)))  # Nueva métrica
Entropy_map_skew = np.zeros((len(P_values), len(K3_values)))  # Nueva métrica
Entropy_map_kurt = np.zeros((len(P_values), len(K3_values)))  # Nueva métrica

#initial state random fixed
seed = 42
rng = np.random.default_rng(seed)
state0 = 0.1 * rng.standard_normal(2 * N_nodes)
T=20


for i, P_val in tqdm(enumerate(P_values)):
    for j, K_val in enumerate(K_values):
        # Simulation using the existing function
        P_vec = np.array([P_val, P_val, P_val])
        t, states = simulate_wc_additive(state0, P_vec, K_val, M, T, dt)

        #deleting the firts 20% of the signal
        start_idx = int(0.2 * len(t))

        E_node = states[start_idx:, :N_nodes]
        I_node = states[start_idx:, N_nodes:]

        # Oscillation detection using the existing function
        is_lc1, _, _, _ = detect_limit_cycle_poincare(E_node[:, 0], I_node[:, 0], dt, threshold_ratio=0.05, period_cv_threshold=0.05)
        is_lc2, _, _, _ = detect_limit_cycle_poincare(E_node[:, 0], I_node[:, 0], dt, threshold_ratio=0.05, period_cv_threshold=0.05)
        is_lc3, _, _, _ = detect_limit_cycle_poincare(E_node[:, 0], I_node[:, 0], dt, threshold_ratio=0.05, period_cv_threshold=0.05)
        oscillation_map[i, j] = int((is_lc1==True) and (is_lc2==True) and (is_lc3==True))


        # Remove transient
        E = states[start_idx:, :N_nodes]

        # Data matrix for higher-order metrics
        X = E.copy()

        # Covariance matrix
        Sigma = np.cov(X.T)

        # Higher-order metrics using the existing functions
        TC_map[i, j] = TC_gauss(Sigma)
        DTC_map[i, j] = DTC_gauss(Sigma)
        Entropy_map[i, j] = entropy_gauss(Sigma)  # Cálculo de la entropía
        skew, kurt = dev_gauss(X)  # Cálculo de la entropía
        Entropy_map_skew[i, j] = skew
        Entropy_map_kurt[i, j] = kurt
        if X.shape[1] == 3:
            Cumulant_map[i, j] = cumulants(X)
            PowerCorr_map[i, j] = powercorr(X)

    if (i + 1) % 5 == 0:
        print(f"Progress: {i+1}/{len(P_values)}")

# Visualization using the existing function
#plot_oscillation_map(oscillation_map, P_values, K_values, title="Oscillation Detection (P-K)")


#cleaing the matrix
TC_clean = clean_rowwise_signed(TC_map, n_std=2)
DTC_clean = clean_rowwise_signed(DTC_map, n_std=2)
Cumulant_clean = clean_rowwise_signed(Cumulant_map,n_std=2)
PowerCorr_clean = clean_rowwise_signed(PowerCorr_map,n_std=2)
Skew_clean = clean_rowwise_signed(Entropy_map_skew,n_std=2)
Kurt_clean = clean_rowwise_signed(Entropy_map_kurt,n_std=2)

# visualization the matrix cleaned
# plot_metrics_contour(, P_values, K3_values, title="Oscilation detection")
plot_lines_with_area(oscillation_map, x_values=K3_values, title="Oscilation Detection")

plot_lines_with_area(TC_clean, x_values=K3_values, title="Total Correlation (TC) without Noise")
plot_lines_with_area(DTC_clean, x_values=K3_values, title="Dual Total Correlation (DTC) without Noise")
plot_lines_with_area(Cumulant_clean, x_values=K3_values, title="Third-Order Cumulant without Noise")
plot_lines_with_area(PowerCorr_clean, x_values=K3_values, title="Triple Power Correlation without Noise")
plot_lines_with_area(Skew_clean, x_values=K3_values, title="Skew with Noise")
plot_lines_with_area(Kurt_clean, x_values=K3_values, title="Kurt with Noise")


4it [01:09, 17.32s/it]


---
## **6. Model with Stochastic Noise**
We add stochastic noise to the excitatory population (E) using the Euler-Maruyama method.

To include biological noise, we extend the excitatory population dynamics using the **Euler-Maruyama method**:

$$
E_i(t + \Delta t) = E_i(t) + \frac{\Delta t}{\tau_E} \left(-E_i(t) + S\left(c_{EE} E_i - c_{IE} I_i + P + K \sum_{j} M_{ij} E_j\right)\right)+ \frac{\sigma_E \sqrt{\Delta t}}{\tau_E} \xi_i
$$

- $(\sigma_E)$: Noise intensity for the excitatory population.
- $(\xi_i \sim \mathcal{N}(0, 1))$: Gaussian white noise.

---
## **7. Oscillation Detection in $(P \times K$) Space** with noise
We perform a parameter sweep to detect oscillations in the $(P \times K$) space.


In [7]:
P_values = [3,4,5,6]
K3_values = np.linspace(0.0, 1.0, 20)

# Matrix to store oscillation detection
oscillation_map_noise = np.zeros((len(P_values), len(K_values)), dtype=int)

# Matrices to store metrics with noise
TC_map_noise = np.zeros((len(P_values), len(K_values)))
DTC_map_noise = np.zeros((len(P_values), len(K_values)))
Cumulant_map_noise = np.zeros((len(P_values), len(K_values)))
PowerCorr_map_noise = np.zeros((len(P_values), len(K_values)))
Entropy_map_noise = np.zeros((len(P_values), len(K_values)))  # Nueva métrica
Entropy_map_skew = np.zeros((len(P_values), len(K3_values)))  # Nueva métrica
Entropy_map_kurt = np.zeros((len(P_values), len(K3_values)))  # Nueva métrica

sigma_E = 0.03
T=450

#initial state
seed = 42
rng = np.random.default_rng(seed)
state0 = 0.1 * rng.standard_normal(2 * N_nodes)



for i, P_val in tqdm(enumerate(P_values)):
    for j, K_val in enumerate(K_values):
        # Simulation with noise using the existing function
        # t, states = simulate_wc_stochastic(state0, P_val, K_val, M, T, dt, sigma_E=sigma_E, higher_order=False)
        P_vec = np.array([P_val, P_val, P_val])
        t, states = simulate_wc_stochastic_additive(state0, P_vec, K_val, M, T, dt, sigma_E = sigma_E, higher_order=False)

        start_idx = int(0.2 * len(t))

        E_node = states[start_idx:, :N_nodes]
        I_node = states[start_idx:, N_nodes:]

        # Oscillation detection using the existing function
        is_lc1, _, _, _ = detect_limit_cycle_poincare(E_node[:, 0], I_node[:, 0], dt, threshold_ratio=0.05, period_cv_threshold=0.05)
        is_lc2, _, _, _ = detect_limit_cycle_poincare(E_node[:, 0], I_node[:, 0], dt, threshold_ratio=0.05, period_cv_threshold=0.05)
        is_lc3, _, _, _ = detect_limit_cycle_poincare(E_node[:, 0], I_node[:, 0], dt, threshold_ratio=0.05, period_cv_threshold=0.05)
        oscillation_map_noise[i, j] =  int((is_lc1==True) and (is_lc2==True) and (is_lc3==True))

        
        # Remove transient
        E = states[start_idx:, :N_nodes]

        # Data matrix for higher-order metrics
        X = E.copy()

        # Covariance matrix
        Sigma = np.cov(X.T)

        # Higher-order metrics using the existing functions
        TC_map_noise[i, j] = TC_gauss(Sigma)
        DTC_map_noise[i, j] = DTC_gauss(Sigma)
        Entropy_map_noise[i, j] = entropy_gauss(Sigma)  # Cálculo de la entropía
        skew, kurt = dev_gauss(X)  # Cálculo de la entropía
        Entropy_map_skew[i, j] = skew
        Entropy_map_kurt[i, j] = kurt
        if X.shape[1] == 3:
            Cumulant_map_noise[i, j] = cumulants(X)
            PowerCorr_map_noise[i, j] = powercorr(X)

    if (i + 1) % 5 == 0:
        print(f"Progress: {i+1}/{len(P_values)}")

# Visualization using the existing function
# plot_oscillation_map(oscillation_map_noise, P_values, K_values, title="Oscillation Detection (P-K) with Noise")


#cleaing the matrix
TC_clean = clean_rowwise_signed(TC_map_noise, n_std=2)
DTC_clean = clean_rowwise_signed(DTC_map_noise, n_std=2)
Cumulant_clean = clean_rowwise_signed(Cumulant_map_noise,n_std=2)
PowerCorr_clean = clean_rowwise_signed(PowerCorr_map_noise,n_std=2)
Skew_clean = clean_rowwise_signed(Entropy_map_skew,n_std=2)
Kurt_clean = clean_rowwise_signed(Entropy_map_kurt,n_std=2)

# visualization the matrix cleaned
plot_lines_with_area(oscillation_map_noise, x_values=K3_values, title="Oscilation Detection")

plot_lines_with_area(TC_clean, x_values=K3_values, title="Total Correlation (TC) without Noise")
plot_lines_with_area(DTC_clean, x_values=K3_values, title="Dual Total Correlation (DTC) without Noise")
plot_lines_with_area(Cumulant_clean, x_values=K3_values, title="Third-Order Cumulant without Noise")
plot_lines_with_area(PowerCorr_clean, x_values=K3_values, title="Triple Power Correlation without Noise")
plot_lines_with_area(Skew_clean, x_values=K3_values, title="Skew with Noise")
plot_lines_with_area(Kurt_clean, x_values=K3_values, title="Kurt with Noise")


4it [08:10, 122.56s/it]
